In [1]:
import duckdb
import pandas as pd
import os
from datetime import datetime


In [2]:
con = duckdb.connect(database = 'dados_duckdb.db', read_only = False)

In [7]:
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-10-08 11:40:12.090243
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-10-08 11:40:12.090243
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-10-08 11:40:12.090243


# Ingestão na tabela da Camanda Bronze

In [8]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos(
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP
        )
""")

In [9]:
con.execute("""INSERT INTO bronze_produtos SELECT * FROM df""")

In [13]:
resultado = con.execute("SELECT * from bronze_z0019").fetchdf()
resultado.head(6)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-10-08 11:36:38.957489
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-10-08 11:36:38.957489
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-10-08 11:36:38.957489
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-10-08 11:40:12.090243
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-10-08 11:40:12.090243
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-10-08 11:40:12.090243


In [11]:
con.execute("ALTER TABLE bronze_produtos RENAME to bronze_z0019")

In [14]:
con.close()